# HDAR Host B — Cross-Platform Continuation Proof

This notebook runs the HDAR Host B protocol on Google Colab.

**What it does:**
1. Installs the `cryptography` package
2. Writes the embedded `run_host_b.py` runner to disk
3. Executes it — restoring the capsule, verifying the Ed25519 signature, running the pipeline, sealing Epoch 2
4. Downloads the Host B report and E2 capsule

**Platform evidence:** Colab runs on Linux (Ubuntu). This is real platform separation from Host A (macOS).

In [ ]:
!pip install cryptography -q
import platform
print(f'Platform: {platform.platform()}')
print(f'Python: {platform.python_version()}')

## Write runner script

The `run_host_b.py` script has the Epoch 1 capsule embedded as base64.

In [ ]:
runner_code = '#!/usr/bin/env python3\n"""HDAR Host B — Self-contained cross-platform runner.\n\nThis script embeds the Epoch 1 capsule as base64. It runs on ANY platform\n(Colab, Codespaces, E2B, local Linux, etc.) and performs:\n\n1. Extract the embedded capsule\n2. Verify the Ed25519 owner signature (using embedded public key)\n3. Restore the workspace exactly\n4. Execute the 5-stage deterministic pipeline\n5. Seal Epoch 2 successor capsule\n6. Write host_b_report.json with platform-specific evidence\n\nUsage:\n    python3 run_host_b.py --out ./host_b_output --host-label my-platform\n\nNo dependencies beyond Python 3.8+ and the `cryptography` package.\n    pip install cryptography\n\nThe capsule, owner public key, and Host A platform are embedded by host_a_seal.py.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport socket\nimport sys\nimport tarfile\nimport tempfile\nimport time\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n# ============================================================================\n# EMBEDDED ARTIFACTS — filled in by host_a_seal.py\n# ============================================================================\nEMBEDDED_CAPSULE_B64 = "H4sICME0YWoC/3RyYW5zcG9ydF9jYXBzdWxlX2Vwb2NoXzEudGFyAO19a5PcNpLgfNav4PVEnLtjVVUk+NaeJlbWw9asNNa6NfZt6DoqQBKsoopFUnx0d8mni/sR9wvvl1wmAJIgq1qkbK3Xt9vcHUsCEq/MRL6QQIW0qJqUrVmRh9u1sWLZdVLm2Z5l9XpPsyRmVb18X+XZn379p8PnWBb/E77Rn7ZrErctE+WGY1jkT5r+p9/ha6qaljD8n/5zfr880LSzJAMkpCmL1gUNd3TDqrNH2juogToahixlJa3Z48fG0rCW+tlDWZNErFzIto8f60ui1uVRVj1+DOCDwjhJGRQTe2kMyre0KA7swALoCuuXztJQq+u6ePzYxOGJUlwlm4ymOK/BtPLqQ5rgfGFK6vAp2wdJyJfhLa2+uNjSCibAF2gq46Y1TcrHj+2lrXZSb0pabLFzw13aXfn+Q4GgavuMhnkW0QXN8mzRVIBX7Z80xMCj1WqVF/Vqn2QJBzFXAjJIo9WJZmvDdSzHdBzL0x2yusnL3fEgTb1Vuv97xcpqxUICPbByFZZ5Xvddh2neRLzJoioAVzCASyzfMDxi2dbpAcI0WQS0Yl8wiGiAneumZ3umYzjmqPMsr4G1okWUh4hSXaFLV1cfClbNQ92gCaLNcC3Lt3zi66OB67S0FsWh3uaZuSibrE72DPnVX5o90CHJsQzYrqdqUURJWQmms5TSLC84yxlKaRluq4KFyty3+Z6talrtYHK2ZyDSXc+yjcEyZLMOxPN10x/Ov9yE+b5IGbK5Cbuln3RV1/mO4eYzAZ39dqkOWVhscIamuvfqGhdDnMGGbKIkz4tFWldiX5NBTcloxHej2qLONyxbALNmdbilNbZTt4esD6HxuOoacOmqnQW03AlM2n1RxWwPBYPRkyIIy0NR465TdnnAYKwkbtIqbwpLUq/DTpCkaULLCMtJT6ggyQUrIFk9ty8G3VDSAy7WU4ZIk2zHSoT1ldI8rfNM5dOiTK6BG1fXtFzFeRrhZtkZK1PfVzfBbeWWxsb5eJN5Bys+1DbXfZti9XZFg2rtu1vT2mVh4MttJbtHnjBMz3RxMw1FQZDXucn50lyaRC0VWB9XlDRkt4JLOpQumLZJ6n9AeVvBEuDv2yZYAqut8mtQAlmUl6tNeii2ebWEyn8y/MC3TZc4HmOhwxzTC3RiB64XRhG1dDNmEdUtN/4z22weB2WeputrUBt5Snesn0hepwnOT1EgojCM42TOxu+huaj0HQNMCN0zBvj5tsxvsoSzn60wEZSC8ALOzHcJM6XS6KubJI1wbooOCGm45bqy3nJ+WRrDqjrPUYmh3uiwHYa3NTKdvbScrgwV6wEBlf0bshLYF9BBdNiUsPW6PTAXGS0aDA8MK9c0PHuABtidZcRgMu5S2RhYWrF6keXlnqbJR1bOGuqoVStzPUu37aGwD3cfN2IHdxsMVEQIWx20cY9xXraIkuiQN3tGMy6xxtVpziWZovZFeZE2myTj0hn/jwxrS1akoru+WZ5kYVNeA1sQQ5UjYZ7mJd1TMYwzKOajg8RU+AS2yF7Iyh6liKYpbcmBUEMalk4M3wZxz5EmkVyV4aC7Radda2CePavLgzKC0uUJQBQdtgeiw/Ncf6jkx03yKTGWfVy9dwrvEG9vjf31Wt+U7+3b1HJsfTsQYyzYhZV9uN06d0wu5xINlm2apg7cemJa8N8atMqiLsFWP8In36CIziBvqmpVQBnjoipkUr0vOEje1IMpDHqFORDP0U3iWcTzTswhTYI93QdAkDy9HuyNp4/u2Bqn2nHpBCaWY9jEFNtydURiaYMvtiBuQdNsZm3Eky1xOIJmo04s1z9eVmvur6sadPr+V43VNZWDgVbyPetoMABrwppvf9D++lENWmzQibAN1Oo6b8riIEzzbms9LelNaj15ieC+ssvRHsi5fX6Y3HgKrGoEg3r1iaEPVUdYVRVsI1zAwNcIa2C0KoXdQVC4KzZCeAhTNBFARBBFShxQN3wU4qmDjSgYhKyuRpYMFLMC5Ws5GjVitBBFZl8UNBuBJm9JjL4YVD+tc+7FKPOIWAyoiG73qUB4V/GMFSUL0YQeOUN/1p5HYA0FKdOkx6fdgHGgZbkGfF0leaYhsUDDa+cRK2DfVcKE0y8U20KSAjyxYvUMtHGa06hacaewm1oS49xK3txUMBolwstUVEWUVDuucUc6FMphJl9BVIiOuHSwfVCkNngSA8YAJ7c1HIlKfnBnwELE4RechFitaAuoBiM1FdZ1P+0m3EWBMDa6QkYCbGwq9GdhVHG9ZCi8zyLYjDUykQuU9rriBPwSPo6nwCbZrUBlP8yeIg5ycD0ES/hKTZIurkG9R4KTiKo/WRbmwGWC1jPMyAp2eV4uUpxuOfwXNyhBMzmOweyAGQz+boYmjcCDY65BPNfQme8EoR2HETcoWbZJGfju/w7DJv9eo65Tmm0akLwdBeo17GNkNcFkPWnAu6dhmINvy8lj9nYXVmHQQ/DFoMWOHURfWGOMaqqRfMbiMi1wYDIsbcW5M56R5PtBpATK1zRIhJDqoW8Z7pJsI7rvoGNb8Dk38DroGDwKLt+5CzqaUQxeMS0SgQZl02A52P432wT87VK4IIZaiZHPCgQM2oHQozFs2jRJxPtUNmfMWNRKbVy80beI90JCKxhBVKc52sDmkiiUeJHSaid87G6LxliGHnQleiZKRR00ccyFJobWYBHd9o1BKkuHBMxY8mWuXtCAPEg4rwaeQe3AZBHyp+s7lq0z5uiG4cYu0wNq2eBvgBnHeTUuURc0WcbKbh5l/pFlURLWc2yMHlo4FLbuOCCBnYHwFUCwKWqh93qsVhg+kS6UIqziJgsPfJe0JRtQdQvOGSg17W6DbDZNLCVsD5vUC5g4Mgw4Ejn2Y7mDahTe1oDm3yX1G6kfkJZWR5ZNnm9SJrxvfViI85GBEhC1es8ASrVUV+DWSGY3fOu4nwYdVJil31NdqVog6cFAJSMPSwXJ8b8AM7IGJIgII7ZTdY4mIOqBFiwUE9HvAgHFAvuWI4kcw5ShScIRjWVdyapmj2bJYs+ihEoDcwQFGKsW6KEByooyr3MuPVwVYyWNMNbnDFvz0nWP5UELBryHfrSpxgY2YDdcH4Q6JErhh1F8Y1MWIQ4Ia1L0pigFdNC6qcaVW8Pg/Sp43nKzU+HwLbsNDjWrRvTaxot2hQMDCspvWS0sjr4QTXsRIOv7BVZpo0h6bxsMOEhZMJZLoTNQMFjOLQ+i6BZeuKgqNjJJts0GfPlNTEO2BtHEBb6yHbcNnk995EHbAU4OUUkX7WTNPsi3PYCMj8GnZ5zOaguoSJMPUpqDmHROVLV7rop2wmzrO0i4tawcWSRRRjkz9yXy1GAg/RNULyMGermHjfDyB7733R4ZCRYDa6ASYdxNUpac7Iu8rIEQsAlqiu4EX6FxAmINGwY8qpBxJaIEqIQUjpPNyMBLigNYshlLUUQOy4VcA+FiHxWvi8MGzxCrdQpqvGwDMh1UXUVgxrAyb6qRqn5PSxrmyxCUXcW+VhjCd6PSapwsNKRhPxyFhyBswzFN2zLdgY5pATFacFtPOZVDaFRdvuW6HnQ7imxIQFRIfJ/M7LiDx6490/NNx/W8oVZ8D3JwfPT11yR7T4nQQR1vv09q6aEqZH2/Z1Ub2FT4530eSC2guCbIveAW0Gpk/WM59BFuv4IH1vUlnTDLth3DGB7mcJg8AZyXXyn6HkRJ5Bop+7BRpyGGQGbBGJoFFo/jHE2kZPEYd4oNCULVOVWDSjJM4gRcb3ClK2662EtFb79vAD8weKuLvJOVXOgN292IQIcgXecWgi1fzgz3SFDkONMxXVP3wTcZLHuX3CQivjUKlO/yfdAI37yTMyk/3Blop5R+PCwwFCDYsQt4t8Gz4jBnnjLOJiM5jmkYhukZgEYRZus7U7ovc+lIK+RKk30iYjGK4kpTuqfrsCjWrdhDk6l33dP0et+eOatGYZpvmrIRXlMPzOMujrpl0o/8qMrqF7+n4TaXew7UW6fFXtNdLvFHethyF+U32QIs1GIy8jWERly5um44lme6w6jwawBsiksaMxG2V+YGah9PXAIuUvSer/qaRQKWecZG54f7kEeuVFdnHzVlOovACMjPRG2bwB++NYzFfEmIas/2AVgJabqfHaXSyfrJy/UT9C6rVd++WwbLGhx21kokLOLeNj1iOr5Nhgpin0T5KOi5T2/HdhYULcQSTGMAyA2B9Ag8j5qU9fkLfh8R2IPoWKBCGKujuxcxaMHdNIMYtmU4owyC1yDr2csszgWq+2nm1wkrDiPvfl/spfpRtNK+2qBR+pWsAT1yzU1hWnTd7gjRvVRByGK6b3sOsT1jKOn2TVon6JXi9lXCprwcPAswq0SIRFc88IyWN1sq0lpUpZwFrTAfCqAsiPE4jR9sG0pCRMaqesGP8ZOccAemp2zGapznrUgG6OaVbU0hqbqtC8y8Bbs5W7Smp+pvZWm9kzu6L2tAaHJ70x4UilAGUSRT7yuaqnTPuewUFmtfuGcbPAKKhbXZl2fZbZeMAd0rojQvWEYT6ROPSvsIDupN3ekPTbE+vF60Utvixs7SJyfrF1tGo5TT8CRgcbhNhRE16L87YRORBHDzDXWCdZNRkVnSF5Yd+o1eeILVA06qdFv0ZZFXdddCHLeIYJiiTQu6j1r7S4HNeOHArsaYVAIOcMJNbn1AY6zLhavm9mVVJdWP29OYm2Jlstlyp9Ex1JnUWxkkZiMOKFhcS3mjlN0W/DxlgJciSdP8BroeHO0WCZC2jfkpGWBQfsvVo6JIi13IvoLZid1Ii5M4nm75wwOtIqU1blLMBJojK1V4cU5mO/Dprjvu9nDTY1elUFezqGpGUxHYUWJDXOvyXALDU5uBD334SmIzYInzwQivrzsk8d5hPaZpOL4DOtkYrSfPw+0oTQak5Eae3huKCQPFsIW2rKnWvVQk9qDdvqjXqGx2SS0CZ7bavJAnQYMAB4/2YFZAJcUJGdQEGOUDy8z+fcKiMCZmv5RrlsFWZv+1asDRKGEb5OWhq1yIym6eFUbMRXyyn3x1CHNM5jLVAIcsXQQJqJzD6UrS1RJ0E/oe64OqvxS8NyVbs2uajrIbigOhRSFjOV5fSssSdzAZZH5CcZUZQgeYw8K1MEoqQbhuPm8OTxoZrwLjwFLaiEyysQhD+Dr/7u/ySNhWWsCCIwydmWp2YXEIacnkkauhlKZJURyduEI5eAr1V9pIjrEz8nX+nrntRhLd405yPMeziO9aQ8MDIMShwixp0wKjJeNavqNbnmO54w65XxhxhWMOwllq3e1xZURBP4fcJVD5S5YvKlbjoc08uThuxBMkwII0bMMi3mjGAnbdxpstZzA6C3ejdAAoxLAhlyMKiRlq2EHW25vDd6z+GdgEOVdXA5xvZBhr3mJkyAsFvGUatmc5zhjrbap1KSN0xomqxTbPdxXPUClRA/NjDLuf0l9/fjtvPu9vuIthmSbRfV8fnV2/ObyuNt/mt6Oz6TeHv9GnqQhoKqjMikYesyiFefAe4/JEXQaWteHX4woef8VZLJ6EKG2SIAG/+TANi8eY1RRYFMGerr4F9E1CXrLyOuFh0CnApsDY6RRcUaTsMiyTov7n5AuAfwjeP50BLQNDcyddFG9LMBhhR73FBBWQBoyffH2+1Y8zZo5C+SfM4Py2TCJhj040AKWO2/aL5g+iHBygJHyC0dkKd9VTHptuSt7NvOaYq/B5uJ9eAFdFs7r8aQZufvoxl+fVn4X7FiizKXFkXN4kU38rElSfc6tgehbfNhXAVdVTnoT9WdCnNGWw/PKynt6tAJtOD/6UlsEkLp+++JvwVqfgYB2cC6bgMIg/Y2540jgDLA9zOgkDPm2Qz2LGp5gweglO+yTcHhy+BEgxc5PAjqhpWM8FQ8toArBk0uaaBzcHlyX7Nm0wIZqHdKZgn/GTq0mw72kBXFHNgHyVh/NoVLLX4vh4Hhyez01Dvnw2B+evX80BymcuYzbzAGiBgVrhbE7BvmW3c8B+fvXkb1Ng3LR8i1dUphkIGeIZq8FDml79M4YLf7rlduAMyGdJFaJLd3h+W7OsmjFAwqcBztNMJOMIP2IGZjStFAawk3sVoHdPyiCpZ4mgZz89e5PSQ0AnMfP8GlTtNFme34aswIG/l4nGk/A8kyvP3uRgyRwmoSU95kykZmVGU2FLohf9efgXYDO/kX72F4BO0uMF+A5sjpR/cclRXE2CTa/9O/jLUww+lnMAxcFHOgt43tgYkpoHeUmvp8yL73lMa7o7nh3xFOR/U7Kn01bLS/RaXrN6m8/Q/C9b/+tNd4lkEjyD3quKpy5NANczyC6hJtnt5Q9ztaoCOaPXy6bEbJsJuORtAwbmqyQo6eSG+2eeOvKaZoCiPY/qfRb8FUUEXILZhy7Da1oU08LlFW2ycDtTJKdJECUVj2JPQ94WU9v5VZLt3oDLCZOeI4jRHEmHHtGXt3i+D1gUsWiSnq9pMsNkByTPAEKj50scdt5grmLlwPP4iYOiMpsUZBzybZ6nAUY4JkD5Wek0zIv/PgdqDjoB7A0r+TlfFrLLLeY+VL+u1XcysWyiaZmEMyb2Ct2Qpp4SAa/ziKWTRvBrfirKWAnaJ0Mb7nqacf5GQbTT9JVMcp+CZvWLy2mYGc6mhJrLsn/L6y5bZ5YS/qFg2bMu1j8Be/lkmlQ/XL7Kp0Tjm1mO8RuWhXMkxZvvn1w+n4LZyuzaaaBJ+QWCNWYYuGJvaDYp2d801Qz74V8aWtYfJ4GScPcqz3dvt80+yECMTmuhH2faQ5c0pmUyU1th6k09p9OQzQkNXYaYsyytp7ngaLyVsyDfikP7CcCkwBAZjx5OQjN8CWHGRMG7KKdFSws2O+jXNkCzbI5h1MJPcvYlyhgQh9wsB+nxBFyYQ5VMsoPgmtnWFCiIkkU/J/X2X/PmS2BnWNYA/5HuZ9AmD5NJ5XrJg6EzkSBvE0wAAR3msDhGPmeAHfYBT2afgMrCuRv7UNVs/yUxbdGiU09TA7zd4iMd81Tf37METYq3h4K9jNDOjJNpYwQTK1Ql+MXwk5uEnzJcNkEFMiNg5ZP2Jtt0o3lW309JWTf4aMEc9P+UzDAKfmbBCV5CzX95+Urk7ajnqDwjJ5OH+MqhWsFKPIWWmS1qgyQ7JPzk27YH5QWrGU9+Gp65F2X+HgyONT9OHN0/eXP4kaff6MNCLsk/ymseypEgbOPdzEPQCkHlrS1TN3SfjM9Aa3AjWEnF8ObSMAdVNV4pUDNbeGGb+DZKDJTZO5h0JNIkMKthnDolYXLYO9dTybEDYJ5lb9u+4xPPJONVINz7vGKjCxyyRqQE0rIWJ8omGdU3Kd4aGaZoifRDTNDIQkkD01Fq6+rW5Ev0lcIbUL3CPhn19VGeHBscHQpHNt1zHfzlnUEWxb8+wXi0MyTAx/0H6GvwRs+HEu/9Yh56t7CSZrtFsCf2KOf3R1ok0Yvm40f5gpfdN+BpKLaSEY9SCx8VmcNqLSxSSXcM09R13xq+dYIge7YGIxI8Jdwhlq2OFSWVeAZFKRMmp3yKwHTVqg1mQ8g7hT09Reejmz0l+9AA1/IkPFPJmmnLF+KO7SALoavrEynJqT4XmAMVsLQejxmHpu85oxx88PtQWw5eFijfJ0wkdSqSit/jtVQal0VULQp+mU1NuSwrmc7YN61LJh4YUsoaEIvp8kD3s5K7FXCeP2GbNj6G4g13nQK1BBkZfGHPvI3sXtddnejeqPsA2y9EWoOu3nUpm1hcCLWXhtsX7otq9A5NZfI3IWJ5tUbJfKvAmu8uwCmP4GA5qvW8HF+Xw7shmBwl9jbpi5NdUi9SMI+z0a1IqBKJuYaSnVwxPKzM5FWfvhuWguRo9vzigboCGX5byKz58Z1KqL+el4gkILku8HUTNo6pDy/MYOAMHbyFwBmIJW552CojtTBFwkI24lmsKw/iSh5ZOoNF1E17BdHjqWL9ujf7FPjA3I32D38DMD2Iuw499Jal6P9t6X7WkntwsW6DEN0y7OGNpSq5lUTqR0/zm/aKep9/VO33tBBJah0rVhktSH1bj8RsBWZc3CrIHkNZfsMfVQTrcS9sBEVLVHlKMy4ITEe5rFiJhDY+gMpHeVmzCNOSaJKJy+YDzkcrvn07wFDnkDdFlTD+xhIwa7+Q/HYsN6si4smhKPg6zSdMfjwd22SJMNbMQWb8v2XOpni1YV2xfJiuKV9zgPJ+7vgunLwB3k/+X149SfGOl7h4rlwBrz6kmzTHVNZh/j+U7zG+Jg0H78T9lu+S+uiOS8n2YLRo51U5daflKa1CGuHpFpqHFfAsEDZs6vYZxMWee5jd1X2w1vBKcpmyWuR2Kw8fQjm+1sUP7gfZnXg4EtEyWtAkDjnBFLOtqwy3Tba7s7ZqMtqcqG1nghnACpvxl4hSboEviSp46zLhr3sOXh0V7w4Jbhs88Vkd5OUG9VkHwHuTUjGoKgxBNNGQxx48tQ+8/slJJfLn+3LulxUgl2BocVmjr0t2/OXG8ZXMGjbjbS3s5cGzGRwaH10TO1WRBXUuXvZRxR6WJVPmLwfiZi/moOKTXMM8VKzn6df64OLIqVeN5FsyZMDbvHAhHqVZdE/gKJdSOQAVaSgnmvaPzVijGnw3iH0QMtFWajIa5eKqc1/6IdpPIgJg+LNsaFL5vk9GeChpUqfyzSZDSS8dKzJFN6tV8tmsxYZlTD7QhJu9n2IJ+xDfZ7EQz0Qp5inRpsoeULa4YQH6XayljNKiAZMRwxxztFcPLawkAt6b6ZGh8qpv+L2+gTeL762W41vHvHBRpcn+VE21UB9LxNcGjFH9wIjmj5mCCaGbxFXAQNXiE8SFSBGZtcZxI+k/WIYDLD+8uiZg10wJu1iD6+71RyH6xGONfWmK53aj7dpkYcqQgRd4U5RfPRQPPJg9RMSTXRgKRrwQEZUJj76aKvv+PUsiJryvQZI6xh7ZvhBSavB6Q1NygwfxqMgi5Pb2qreSNd1cywtp/bSuAYvCflT9gOY6zfNCip5uld1DUeIBLwVX15xtBjL4Og/z8e2lGxD+gbxXoBTWsMXz8YNCvFi+KT14YOAm3Msz3oG9eBPeJFG9HV1xgM1joh+oPvMLZQL5rTIUvhFR6sWG695gGdxy6aorcZlpVJ7Eh7GU/5mVu4+s2Yj7ZR2L32yZsAQsRbfelLSoR3bqTcVvtAjzryu9HVnWBHP5u35uN0GeV7WIQHUzub0VT0OZKo4OoHS5XlUAD3lTNwHYBjxOPzIt20o+Pg/7t7fjlHd3D/Ui4s9H4fZxerP3Y1IU8uJBR5GPrVkwZ5N3wOINZc/XXc9uQzjQ3xV2esZ3KTt7pJ09Xf797YuFMLX41Wu8gfMI31neO2K2Z3m1BnNkz/qn0Z/REkRhfwEYrwQveZ9t2dCwEPCayInQfpJ3kgXMI+1tw7S/Npmm+Rohj4j3yLS0N8/eaoibf9RQKT26zZoF2KUusCoBZtf/l7H68fmr508un6+f/PjasdZvHd3u32Tu5i7+0a+7ffaRR1RhPY6ji3J5TQ1XDkj44XJB+BshC94V/nfhWAG4nq8BQ4sfBF7ktaG8lPiSpSKWJV1IrOJWnEG0/6nJ4SMtOGhP5EugD7WXWbiEyvM9OBcPtRcs0IjFF/9QM5xHuvuI6Bfau6f4/JmGL/QuPe1KjIU268dcEOz55duzB58e/On+m/eFo99/+Fq/+TD/9x/AO+7/Ln//wbRc8/73H36333/gT+atkwg30Bak5gJs4KpatOJAvOou9hp6/0nW8DOTNX8RBRt9D1pEe6KBIkhhW3Ne0ox/1Hjxt9oe3+0tGbcrH2riWT+m4a1efCHiIW8mG5GlHAasYzCD1hS7xxfyXds2ffA8Xcu2TA5y6pdK1qi5cEKRZTJGIj1wdcujUeB5gWUH1IuIG1sGsX3TJSwI3cgCxeCGhMW27hlBFDgstnQiXuQ943PCCUi1MBrEoy6NvNCAqUW251InDiIWeMyklh86fuCHpuHqoRcYQWwBQBR7phvGtmvGHnNCqW3y4D3PjRF6aIB47ftnT37UVJRrIG3zWDa8Ad9hXTRBmoTrHTvwKfk+/L+hezQ0Hd3wwsDw3RhmxphDTdcOHBLAWmOfhR5sMhCvPqXU9DwX1u+Fg575r2tgqoBYq2HoFvhDrh7HEdFdQmMnBlsccBCGZgTDEovSwAyRTjZzHTNyaRxYTujTwPIC03ZM1woMZsSx5ViMsdg3/dixbSeOfQdQ6MYs8kOoNANbd53A173Q83TDcajunpzYmqabHKzeLddYLCK2LZ9dOONXno4ZI2vStNVa+C5+qmoowfr4sxsJ6PDVdWuUnpUsZGDAdISHaQe6EbgsjqwAsEup4TLDY4wGUch8QolBQmJYJvWMyAxdRokVhLEX4nNQsSec3TPx8k878JIbSnjTbCFl8uq6PRw4E092rbewn9YpDVjKW+V4LLbgBgdYqeEx6K/S52giVQU+utYiD5r/0j9X2f82jCaL27dFkFEsoj/sC0uW8mfyuWnAhQw+bce4cjlT4KotJbaDUA6JHcdlBD7XjixAGwlCautUZ3ijMnYj0yBR5ACmXT8EPmMOWHdxbFhAkiggg16Tjzgl07Nk2aeHv2Le6OCtEkwkXpc8Qb/i009Pzz9kcWiEwIjUcSlsBsIim8WGD64lCwLPZVFkRChtYhRNNoW/E9PXwcuO8KVX88T8wTElv2UBeOMer0WCabo5PWndNwKLeNR2Iw83Zsh0X/eBxUFKmpZlOZ5l2mBAB5T6sYHx0hCWYVKgjB5bbnRi0v5vwnlVhtxUZ+Wye69pzCghtWBuGKYIosgOQ8ekLvEC2HyuCwLYdUDMxT6JTN+IfCv0dcuhwFF+QAxmd4axOmfb8PTfMmu8Mb3cR6fnazogJCwTZGdghbpuUnBLXNeOI9M0w0gnHohMFyQoSEXbCUBuWCC1deZ6IKsZ8+0T8zUsu50u//OqPZYCX6GTVS4l0Ckz3dj2WOzFAYto5DA9NKB306dmANgxGUh1YhJiERDBATgRzIs91/H7m/d1XtN03VIXBPYDHPbTvQX9H8v+l6rua5r/U/a/oRNjbP+D4XRv//8R7f+hSczwMhNXe5KNhAdw9m9pL/8uxpgc5N/JBMOABrDlvjjh/PgjI+3rCvt7gf6fXP4H+Bp9tdL91W81CmfLf+jRGcp/ZM773//8feS/ItFVeS7jL3kmxBg9eyikXS/1sOSkoCJLF6StTYz7MOz/t/vfdFa/1WGZv//J2P6D/Q/g9/v/d/j+LGKMb2m1014lVf3gwZ//rD0XAVztXMR1Lx4stHe3V9pTLhS0zvaQxZcYvpV8pDQnsvm3vLl2pT0fBX5lMW9eNfIG/73I+EPsf4esfmskbv7+d8bnP8QEkXK////4/p+wDab8oQx/C4HyXJvusGh4KtTLlFPnQzSLTp0RiV9J4eFC7nWqxgr3okCmcbcLb2Jg4HnD1lReMlu3vfeg6/YHlRENMU0r1lfxxsqPovPnFZXfcKr7ZEm62ZQMf1ao/+lRWlVJfOjT+DHIzw/j/wAH1Xft/3D1WwOsc/e/QXTXGu9/w7Xv9//vov//y6qpylWQZCu8HSV/EP3B2dkZv72/4Jyvtdum25Ta//3f/6c93h2cT+J2WULrB+IXZzSMIz7UqkP1UMMgRZoED+Iy32sYM8cLHBLsDfxTVIT4Sg2XFVVbGbGYwmTw3fUHD+AfWndGt8buz/GE5uIR310lq5sy46MuI7ydwSthAtDPmv+A3duy4UfOsIVFltr52UN0ZR6dXYDsySp+rFiFScIhL8SAIoy/5j+tJIbTFn/RqroUoyaxllQ8HzsLWTtiXco54Ydl2mP+x1L8ZOL5hTphiZulGEgMsdyy2ygBuVOft9PgMowLn/NOYspRxAkVohXG6Sq1lTjAOsO/nDrEkpPgRdDwHccc/8GD8/RCA+mvpdC10vsSE5jXmN98frHkmdX4J/7qxvnZ/8jOLhAZaVtxJbsHEVND77+c8RXwQykhQXk4D8de89//QNGbsuxcFgJFQLqWldRNsvSdfvXuDAquHuJPh4wrF4as/aSM/a49D+aBsiuYypn+G78zlXxiFJVIQimcK3pNTvChpkxF0o7nTCLEe578iXSAxb0T2EMalEgD2UHPVIDpLAcdutwwQD6s+UIDWKUIf8h1k5cHUYGpmQ074wDYXd+RWIkYfEmLgmXR+S9nAq1d5w8xP3WX5TcZ7hS8WlcJfb5PKrzHCktmaVSdfbro+gVRIaao7I7ynZwHrPA8yeqHWgykry/4FPtK7b9p+rwJdrzQzyjJOEbXoq/BhCo27JUDtl2Wgz2pcGur4rUBHz1SSfmw3V9tUt2Ij8WM1EpeIlApVjZqKgpVAM7m4q+fVGbrbI6Z/BYc1sAaKI96yXqOv6N4McFxouE7wHnHWldXHfY64olu0D5DmfLLp65XaPUQUV5h7+I60bnoc4m5y9X5hSIzeft3UHfFRccIdRypVYOBdf6W6jn8vS23sA5/O/64cqU0F2AJQsF/u0739BZL6G1fgm8JIZicMi9/1/W0WpFO2ow5RzEHP888EqEJNzORFudD/KDyAvTwVSNicDb454ARWlvzJB9w8FPsUON9f4HjMsFnptCWRwl0tk022/bviASObv6vNL/hf/00wTHhHjoWlBxyzTtBoKsOkj+DAMCKDFhhcxAh8N+/aDrfvZquij/R5i+PwSfQH4l1vOsXoXImyoiRYOoaG0u7a8xXPLehrjSU6PlcU5A9LTSi7zToMQv1DsTnOQi7FiIEmeOXndwqQovvYN8hgfj47Wb71LbC3GUoP5PzG/CU8FfOb5B1SpDW8D8K/wvLU6x0PHvp7kzMvXPHZvlr/YcSYE9LzH37RaYocCmMAwBaR8bFVSeHZQV6eoLfOtl8NZS3Me9lIJ+vxruV9pyNBVctVvmvhydBIx3fEMBUGl19Gq2l3R3rdoNjx+LvV8eU6rpri8QmFb//yPEhvNaBX2sjBvqMN7wx8UlSG/OuzyUZe/vxMTfNz8GAX9Jyc/3OuBIWHnBWW3YBm9MQm5PDni3PBCvnTT22RaEIqdNWL/e7KCnPBS+0tjm7Bbyt8500wBG0KIUUOWH9inos5LyEcKqxPnIVivKis7/brt8JRPW2Yd8bhzjHday0M2V8kcB2sbzBKzfCGFZcDtwm+CBoVj8mY9fjQvsHjdvJcnCwgM7js0vuZRnaOe/9AogH0/pmyL3fXH3qDHXJzxd/dMs/7un2eYNY4lvaHxx0DjnjUlIxPqai0ssJMoraz9Mx/pV0JNq56B4JCTP7RpEvSEVp7vOqoWwRNJbC56L3DZA+5ZGW5RpQMf4xTnZs+WOpan8LAKFiL8Y2OO+iV8Co4wQlaU/Jk9amXJOCdDF9AJ5DSNoSkh4Tsu3iBBW7qXyekPRXEtLUzrsRkJYo9WCC3/TC/puri0+aIvtFL2GPrFMWmcTVQLq3i5QdyFZzUBe2qAuPUTfo5wT+2vrPoy/8leiztPN2AMQezO8bRf8Br5+1Jg8P8zweGRw9wsZ2x2BZso8pNIleL9QRj9BVDnEUJ2B9yOl8HkMCZi6WhjT4+v0PqWBr7drBmhmvSMPxapaJ6NzjX8otEuUBCJb1Gi+7rdfa48fa2XqN1sF6fSbMA2Eq3B/V/cc9/wvZ6rdmss+O/2MC6Cj+bxmucR///33yf8SxH5gUCx7W7J0bfqGIpsWWp/oIk+CR5uPl06PUn5Z2nx6MOjTGHQbgoaj9Gb651L07OnSOOyTjDjcUPEC1R8tbev7pDg1y3KE57jBi6XCKpkXuXLPhHfdozUGiZZ/ukFjHHdqTSLTwxv7pDs0TVHEmkWjjE2d3dHiCKu4kEolnLn3ndI/WCbJ400i0l85dHZ6gij+FRKJbS8c63aF9TBVDn0YiuROJzjFVDGOaEx1/6d1BZ+eYLAaZRKKBl7LN0z26x2QxzDlYtO5YtHtMFsOa3s9k6dzRoXeCLPY0K5r+nfvZP0EXZxqLsGj/Diz6J+jiTmIRWMc6zYugLY879KZ5Ub+LLIZ+giz+NBZBcN9BF8M4pgvRZ/CiftcGBMF93OO0brGMpefc0eExWcgM3eIsvdPKyjCPyUJm6Bbi30kX85guxJqBRVAu7ukerRN0mVYuPj6EebpD+wRZZikX964OT5BlWrmYhnWXzjecE3TxZmARtIt9R48n6DKpXQzbvpMs7jFZzBnaBWZ4By96x2Qxp7UL0f27dIHhHdPFnNYulru8Y/v5x1QxJ3WL4dpL947N4h8TxZyhW/Sld1rigB477nBat5g2Pmh7ukfjBFWcWRqa+Hf0eIIq7jQnGktyx6LJCbLM0C3eXaxNyAmy+LM0tHdHj+YxXawZbou3JKc3Cxjcxx1OqxbPuxOJ1jFVrGnVYuODg6c7tI+pYs1xW+40c4h9TBZrjmohdwlu4pwgy6RqIfiM3+n+3BNUmaFZnKV9R38niDJDsRBv6Z42uIl3girTisUD7XyHfPBOEGVar7jG0rxjhv6YJgGNFscuuZjjCUgyNfjCvr/jc//df/ff/Xf/3X/33/13/91/99/9d//df/ff/Xf/3X/33/13/91/99/9d//df/ff/ffH+P4fB8zYTQDIAAA="\nEMBEDDED_CAPSULE_SHA256 = "0d39b5e9f573a35218cb9b6cd891dd01b9d5a3242e0dd788308bf3ba6edf8e6c"\nEMBEDDED_OWNER_PUB = "899899108ac36018cb197fa6fee6a375b62bef5f9ec83472029aaa38875088cf"\nEMBEDDED_HOST_A_PLATFORM = "macOS-26.5.2-arm64-arm-64bit-Mach-O"\n# ============================================================================\n\nCHUNK_SIZE = 1024 * 1024\nSCHEMA = "hdar.transport-capsule/v0.1"\nRECEIPT_SCHEMA = "hdar.receipt/v0.1"\nAGENT_ID = "hdar-cross-platform-agent"\nPROTOCOL_VERSION = "hdar-canonical/v1.0"\n\n# ---------------------------------------------------------------------------\n# Crypto\n# ---------------------------------------------------------------------------\n\ntry:\n    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey, Ed25519PrivateKey\n    from cryptography.hazmat.primitives import serialization\n    HAS_CRYPTO = True\nexcept ImportError:\n    HAS_CRYPTO = False\n    print("FATAL: cryptography package required. Install: pip install cryptography", file=sys.stderr)\n    sys.exit(1)\n\n\ndef generate_keypair() -> tuple[bytes, bytes]:\n    """Generate an ephemeral Ed25519 keypair for Host B signing."""\n    priv = Ed25519PrivateKey.generate()\n    pub = priv.public_key()\n    priv_bytes = priv.private_bytes(\n        encoding=serialization.Encoding.Raw,\n        format=serialization.PrivateFormat.Raw,\n        encryption_algorithm=serialization.NoEncryption(),\n    )\n    pub_bytes = pub.public_bytes(\n        encoding=serialization.Encoding.Raw,\n        format=serialization.PublicFormat.Raw,\n    )\n    return priv_bytes, pub_bytes\n\n\ndef sign_message(priv: bytes, msg: bytes) -> bytes:\n    """Sign a message with an Ed25519 private key."""\n    key = Ed25519PrivateKey.from_private_bytes(priv)\n    return key.sign(msg)\n\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(CHUNK_SIZE), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef canonical_json(data: dict) -> bytes:\n    return json.dumps(data, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode()\n\n\ndef verify_signature(pub_bytes: bytes, message: bytes, signature: bytes) -> bool:\n    try:\n        pub = Ed25519PublicKey.from_public_bytes(pub_bytes)\n        pub.verify(signature, message)\n        return True\n    except Exception:\n        return False\n\n\n# ---------------------------------------------------------------------------\n# Workspace hashing and capsule operations\n# ---------------------------------------------------------------------------\n\ndef hash_workspace(workspace: Path) -> dict:\n    files = []\n    total_size = 0\n    for path in sorted(workspace.rglob("*")):\n        if not path.is_file() or path.is_symlink():\n            continue\n        rel_path = path.relative_to(workspace).as_posix()\n        st = path.stat()\n        entry = {"rel_path": rel_path, "sha256": sha256_file(path), "size": st.st_size, "mode": st.st_mode & 0o777}\n        files.append(entry)\n        total_size += entry["size"]\n    root_material = "\\n".join(f"{f[\'rel_path\']}|{f[\'sha256\']}|{f[\'size\']}|{f[\'mode\']}" for f in files).encode()\n    return {"root_hash": sha256_bytes(root_material), "files": files, "total_size": total_size}\n\n\ndef verify_capsule(capsule_dir: Path, owner_pub_hex: str) -> dict:\n    manifest = json.loads((capsule_dir / "manifest.json").read_text())\n    problems = []\n\n    # Verify manifest hash\n    signing_content = {k: v for k, v in manifest.items() if k not in ("manifest_hash", "owner_signature")}\n    expected_hash = sha256_bytes(canonical_json(signing_content))\n    if expected_hash != manifest.get("manifest_hash"):\n        problems.append("manifest hash mismatch")\n\n    # Verify content blocks\n    missing = 0\n    corrupt = 0\n    for entry in manifest["workspace_manifest"]["files"]:\n        digest = entry["sha256"]\n        blob = capsule_dir / "blocks" / digest[:2] / digest\n        if not blob.exists():\n            missing += 1\n        elif sha256_file(blob) != digest:\n            corrupt += 1\n    if missing:\n        problems.append(f"{missing} content blocks missing")\n    if corrupt:\n        problems.append(f"{corrupt} content blocks corrupt")\n\n    # Verify Ed25519 owner signature\n    sig_valid = False\n    if "owner_signature" in manifest and "owner_public_key" in manifest:\n        if manifest["owner_public_key"] != owner_pub_hex:\n            problems.append("owner public key mismatch")\n        else:\n            sig_valid = verify_signature(\n                bytes.fromhex(owner_pub_hex),\n                manifest["manifest_hash"].encode(),\n                bytes.fromhex(manifest["owner_signature"]),\n            )\n            if not sig_valid:\n                problems.append("owner Ed25519 signature INVALID")\n    else:\n        problems.append("no owner signature in manifest")\n\n    # Verify receipt (try both old and new hash methods for backward compat)\n    receipt = json.loads((capsule_dir / "receipt.json").read_text())\n    receipt_expected_new = sha256_bytes(canonical_json(\n        {k: v for k, v in receipt.items() if k != "receipt_hash" and k != "manifest_hash"}\n    ))\n    receipt_expected_old = sha256_bytes(canonical_json(\n        {k: v for k, v in receipt.items() if k != "receipt_hash"}\n    ))\n    if receipt_expected_new != receipt.get("receipt_hash") and receipt_expected_old != receipt.get("receipt_hash"):\n        problems.append("receipt hash mismatch")\n\n    return {\n        "ok": not problems,\n        "problems": problems,\n        "manifest_hash": manifest["manifest_hash"],\n        "workspace_root_hash": manifest["workspace_manifest"]["root_hash"],\n        "epoch": manifest["epoch"],\n        "owner_signed": "owner_signature" in manifest,\n        "signature_valid": sig_valid,\n        "parent_manifest_hash": manifest.get("parent_manifest_hash"),\n        "file_count": len(manifest["workspace_manifest"]["files"]),\n        "total_size": manifest["workspace_manifest"]["total_size"],\n    }\n\n\ndef restore_workspace(capsule_dir: Path, dest: Path) -> dict:\n    manifest = json.loads((capsule_dir / "manifest.json").read_text())\n    if dest.exists():\n        shutil.rmtree(dest)\n    dest.mkdir(parents=True)\n    for entry in manifest["workspace_manifest"]["files"]:\n        blob = capsule_dir / "blocks" / entry["sha256"][:2] / entry["sha256"]\n        out = dest / entry["rel_path"]\n        out.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(blob, out)\n        os.chmod(out, entry["mode"])\n    restored = hash_workspace(dest)\n    return {\n        "restored_root_hash": restored["root_hash"],\n        "expected_root_hash": manifest["workspace_manifest"]["root_hash"],\n        "exact": restored["root_hash"] == manifest["workspace_manifest"]["root_hash"],\n        "file_count": len(restored["files"]),\n    }\n\n\ndef safe_extract_tar(tf: tarfile.TarFile, dest: Path) -> None:\n    dest_resolved = dest.resolve()\n    for member in tf.getmembers():\n        if member.name.startswith("/"):\n            raise ValueError(f"tar member has absolute path: {member.name}")\n        if ".." in Path(member.name).parts:\n            raise ValueError(f"tar member has path traversal: {member.name}")\n        if member.issym() or member.islnk():\n            raise ValueError(f"tar member is symlink/hardlink: {member.name}")\n        if not (member.isfile() or member.isdir()):\n            raise ValueError(f"tar member is not regular file or dir: {member.name}")\n        member_path = (dest / member.name).resolve()\n        if not str(member_path).startswith(str(dest_resolved) + os.sep) and member_path != dest_resolved:\n            raise ValueError(f"tar member escapes destination: {member.name}")\n    if sys.version_info >= (3, 12):\n        tf.extractall(dest, filter="data")\n    else:\n        tf.extractall(dest)\n\n\ndef seal_epoch_2(\n    workspace: Path,\n    capsule_dir: Path,\n    epoch: int,\n    parent_manifest_hash: str,\n    host_label: str,\n    host_b_priv: bytes | None = None,\n    host_b_pub: bytes | None = None,\n    challenge_nonce: str = "",\n) -> dict:\n    capsule_dir.mkdir(parents=True, exist_ok=True)\n    blocks_dir = capsule_dir / "blocks"\n    blocks_dir.mkdir(parents=True, exist_ok=True)\n\n    workspace_manifest = hash_workspace(workspace)\n    for entry in workspace_manifest["files"]:\n        src = workspace / entry["rel_path"]\n        digest = entry["sha256"]\n        dest = blocks_dir / digest[:2] / digest\n        if not dest.exists():\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(src, dest)\n\n    # Generate Host B environment manifest and bind its hash into E2\n    # (audit defect 7: environment manifest was not evidence-bound)\n    import subprocess as _sp\n    def _pip_freeze() -> list[str]:\n        try:\n            r = _sp.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=10)\n            return r.stdout.strip().split("\\n") if r.returncode == 0 else []\n        except Exception:\n            return []\n    # GPU detection / emulation — extract GPU from CPU if no hardware GPU\n    gpu_manifest = None\n    try:\n        from gpu_emulator import GPUEmulator\n        _gpu = GPUEmulator()\n        gpu_manifest = _gpu.manifest()\n        print(f"  GPU: {_gpu.summary().split(chr(10))[0]}")\n    except Exception as e:\n        print(f"  GPU detection skipped: {e}")\n    host_b_env_manifest = {\n        "python_version": sys.version,\n        "platform": platform.platform(),\n        "processor": platform.processor(),\n        "machine": platform.machine(),\n        "os_uname": list(platform.uname()),\n        "hostname": platform.node(),\n        "installed_packages": _pip_freeze(),\n        "gpu_manifest": gpu_manifest,\n    }\n    host_b_env_manifest_hash = sha256_bytes(canonical_json(host_b_env_manifest))\n\n    manifest = {\n        "schema": SCHEMA,\n        "protocol_version": PROTOCOL_VERSION,\n        "agent_id": AGENT_ID,\n        "epoch": epoch,\n        "parent_manifest_hash": parent_manifest_hash,\n        "created_at": time.time(),\n        "source_host_label": host_label,\n        "source_host_platform": platform.platform(),\n        "objective": "Cross-platform HDAR continuation proof — Epoch 2 sealed by Host B",\n        "continuation_point": f"Host B ({host_label}) restored E1, executed pipeline, updated agent state, sealed E2.",\n        "workspace_manifest": workspace_manifest,\n        "host_b_environment_manifest_hash": host_b_env_manifest_hash,\n    }\n    if challenge_nonce:\n        manifest["challenge_nonce"] = challenge_nonce\n    if host_b_pub is not None:\n        manifest["host_b_public_key"] = host_b_pub.hex()\n\n    # Bind the receipt hash into the manifest (audit defect 6: receipts were\n    # self-hashed only, not anchored into the signed manifest).\n    # Receipt hash is computed over a stable subset (excluding manifest_hash\n    # and receipt_hash) to avoid circular dependency. See host_a_seal.py for\n    # the full explanation.\n    receipt = {\n        "schema": RECEIPT_SCHEMA,\n        "event": "capsule_sealed_after_host_b_continuation",\n        "agent_id": AGENT_ID,\n        "epoch": epoch,\n        "source_host_label": host_label,\n        "source_host_platform": platform.platform(),\n        "manifest_hash": "",  # filled after manifest hash is known\n        "workspace_root_hash": workspace_manifest["root_hash"],\n        "timestamp": time.time(),\n    }\n    if host_b_pub is not None:\n        receipt["host_b_public_key"] = host_b_pub.hex()\n    receipt["receipt_hash"] = sha256_bytes(canonical_json(\n        {k: v for k, v in receipt.items() if k != "receipt_hash" and k != "manifest_hash"}\n    ))\n\n    # Bind receipt_hash into manifest, compute manifest hash\n    manifest["receipt_hash"] = receipt["receipt_hash"]\n    signing_content = {k: v for k, v in manifest.items() if k not in ("manifest_hash", "host_b_signature")}\n    manifest["manifest_hash"] = sha256_bytes(canonical_json(signing_content))\n\n    # Fill in manifest_hash in receipt (informational linkage)\n    receipt["manifest_hash"] = manifest["manifest_hash"]\n\n    # Host B signs the manifest hash\n    if host_b_priv is not None and host_b_pub is not None:\n        signature = sign_message(host_b_priv, manifest["manifest_hash"].encode())\n        manifest["host_b_signature"] = signature.hex()\n\n    (capsule_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True))\n    (capsule_dir / "receipt.json").write_text(json.dumps(receipt, indent=2, sort_keys=True))\n    # Write Host B environment manifest alongside the E2 capsule\n    (capsule_dir / "environment_manifest.json").write_text(\n        json.dumps(host_b_env_manifest, indent=2, sort_keys=True) + "\\n"\n    )\n\n    return manifest\n\n\ndef execute_pipeline(workspace: Path) -> dict:\n    worker_path = workspace / "src" / "worker.py"\n    if not worker_path.exists():\n        return {"ok": False, "reason": "src/worker.py not found"}\n    import subprocess\n    result = subprocess.run(\n        [sys.executable, str(worker_path), str(workspace)],\n        capture_output=True, text=True, timeout=60,\n    )\n    if result.returncode != 0:\n        return {"ok": False, "reason": f"worker.py exited {result.returncode}: {result.stderr}"}\n    output_path = workspace / "output" / "final_report.json"\n    if not output_path.exists():\n        return {"ok": False, "reason": "worker.py did not produce output/final_report.json"}\n    output = json.loads(output_path.read_text())\n    output_hash = sha256_bytes(canonical_json(output))\n\n    stage_chain = []\n    for sname in ["parse", "filter", "aggregate", "classify", "report"]:\n        sfile = workspace / "output" / f"stage_{sname}.json"\n        if sfile.exists():\n            sdata = json.loads(sfile.read_text())\n            stage_chain.append({"stage": sname, "hash": sdata.get("stage_hash", ""), "parent_hash": sdata.get("parent_hash")})\n\n    return {\n        "ok": True,\n        "pipeline": "multi_stage_analysis_pipeline",\n        "stages_completed": output.get("metadata", {}).get("stages_completed", 0),\n        "output_hash": output_hash,\n        "stage_chain": stage_chain,\n        "stdout": result.stdout,\n    }\n\n\ndef utc_now_iso() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description="HDAR Host B — Cross-platform runner")\n    ap.add_argument("--out", default="./host_b_output", help="Output directory")\n    ap.add_argument("--host-label", default="", help="Label for this host (auto-detected if empty)")\n    ap.add_argument("--operator", default="", help="Operator identity (optional)")\n    ap.add_argument("--challenge-nonce", default="", help="Verifier-issued challenge nonce (optional, for challenge-response freshness)")\n    args = ap.parse_args()\n\n    if not EMBEDDED_CAPSULE_B64:\n        print("FATAL: This script has no embedded capsule. Run host_a_seal.py first to generate it.", file=sys.stderr)\n        return 1\n\n    runner_start = time.time()\n    runner_start_utc = utc_now_iso()\n    machine_nonce = str(uuid.uuid4())\n    machine_hostname = socket.gethostname()\n    host_label = args.host_label or f"{machine_hostname}-{platform.system().lower()}"\n\n    out_dir = Path(args.out).resolve()\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    print("=" * 60)\n    print(f"HDAR Host B — Cross-platform continuation proof")\n    print(f"Host label: {host_label}")\n    print(f"Platform: {platform.platform()}")\n    print(f"Python: {sys.version.split()[0]}")\n    print(f"Hostname: {machine_hostname}")\n    print(f"Nonce: {machine_nonce}")\n    print(f"Started: {runner_start_utc}")\n    print("=" * 60)\n\n    # 1. Decode and verify embedded capsule\n    print("\\n[1/6] Decoding embedded capsule...")\n    capsule_bytes = base64.b64decode(EMBEDDED_CAPSULE_B64.encode())\n    capsule_hash = sha256_bytes(capsule_bytes)\n    print(f"  Capsule SHA-256: {capsule_hash}")\n    if capsule_hash != EMBEDDED_CAPSULE_SHA256:\n        print(f"  FATAL: Capsule hash mismatch! Expected {EMBEDDED_CAPSULE_SHA256}", file=sys.stderr)\n        return 1\n    print(f"  Hash verified: matches embedded checksum")\n\n    # 2. Extract capsule\n    print("\\n[2/6] Extracting capsule...")\n    tar_path = out_dir / "transport_capsule_epoch_1.tar.gz"\n    tar_path.write_bytes(capsule_bytes)\n    with tarfile.open(tar_path, "r:gz") as tf:\n        safe_extract_tar(tf, out_dir)\n    print(f"  Extracted to: {out_dir}")\n\n    # Find capsule directory\n    capsule_epoch_1 = None\n    for candidate in ("capsule_epoch_1", "capsule"):\n        cdir = out_dir / candidate\n        if (cdir / "manifest.json").exists():\n            capsule_epoch_1 = cdir\n            break\n    if capsule_epoch_1 is None:\n        for d in out_dir.iterdir():\n            if d.is_dir() and (d / "manifest.json").exists():\n                capsule_epoch_1 = d\n                break\n    if capsule_epoch_1 is None:\n        print("FATAL: Could not find capsule directory with manifest.json", file=sys.stderr)\n        return 1\n    print(f"  Capsule directory: {capsule_epoch_1.name}")\n\n    # 3. Verify capsule (signature, hashes, blocks)\n    print("\\n[3/6] Verifying Epoch 1 capsule...")\n    verification = verify_capsule(capsule_epoch_1, EMBEDDED_OWNER_PUB)\n    if not verification["ok"]:\n        print(f"  FATAL: Capsule verification failed: {verification[\'problems\']}", file=sys.stderr)\n        return 1\n    print(f"  Manifest hash: {verification[\'manifest_hash\']}")\n    print(f"  Workspace root hash: {verification[\'workspace_root_hash\']}")\n    print(f"  Owner signature valid: {verification[\'signature_valid\']}")\n    print(f"  Files: {verification[\'file_count\']}, Size: {verification[\'total_size\']}B")\n    print(f"  Host A platform: {EMBEDDED_HOST_A_PLATFORM}")\n    print(f"  Platform separation: {\'YES\' if platform.platform() != EMBEDDED_HOST_A_PLATFORM else \'NO (same platform)\'}")\n\n    # 4. Restore workspace\n    print("\\n[4/6] Restoring workspace...")\n    restored_workspace = out_dir / "restored_workspace"\n    restore_report = restore_workspace(capsule_epoch_1, restored_workspace)\n    if not restore_report["exact"]:\n        print("FATAL: Workspace restoration was not exact!", file=sys.stderr)\n        return 1\n    print(f"  Restoration exact: True")\n    print(f"  Files restored: {restore_report[\'file_count\']}")\n\n    # 5. Execute pipeline\n    print("\\n[5/6] Executing 5-stage pipeline...")\n    task_result = execute_pipeline(restored_workspace)\n    if not task_result["ok"]:\n        print(f"FATAL: Pipeline execution failed: {task_result.get(\'reason\')}", file=sys.stderr)\n        return 1\n    print(f"  Pipeline output hash: {task_result[\'output_hash\']}")\n    print(f"  Stages completed: {task_result[\'stages_completed\']}")\n    for stage in task_result["stage_chain"]:\n        print(f"    {stage[\'stage\']}: {stage[\'hash\'][:16]}...")\n\n    # 5b. Update agent state to reflect completion (semantic continuation)\n    print("\\n[5b/6] Updating agent state for Epoch 2...")\n    agent_state_path = restored_workspace / "agent_state.json"\n    agent_state = json.loads(agent_state_path.read_text())\n    agent_state["epoch"] = 2\n    agent_state["task_completed"] = True\n    agent_state["status"] = "completed_on_host_b"\n    agent_state["previous_manifest_hash"] = verification["manifest_hash"]\n    agent_state["completed_at_utc"] = utc_now_iso()\n    agent_state["completed_on_platform"] = platform.platform()\n    agent_state["next_action"] = "Epoch 3: Transfer E2 to next host or verify lineage."\n    agent_state_path.write_text(json.dumps(agent_state, indent=2, sort_keys=True) + "\\n")\n    print(f"  agent_state.epoch = {agent_state[\'epoch\']}")\n    print(f"  agent_state.task_completed = {agent_state[\'task_completed\']}")\n    print(f"  agent_state.status = {agent_state[\'status\']}")\n\n    # 5c. Update todo.md to mark Epoch 2 work complete\n    todo_path = restored_workspace / "todo.md"\n    todo_path.write_text(\n        "# HDAR Task List\\n\\n"\n        "## Epoch 1 (Host A)\\n"\n        "- [x] Create workspace\\n"\n        "- [x] Seal capsule\\n\\n"\n        "## Epoch 2 (Host B)\\n"\n        "- [x] Execute pipeline\\n"\n        "- [x] Seal successor\\n"\n        "- [x] Update agent state\\n\\n"\n        "## Epoch 3 (Next Host)\\n"\n        "- [ ] Restore E2 capsule\\n"\n        "- [ ] Continue work\\n"\n    )\n    print(f"  todo.md updated: Epoch 2 marked complete")\n\n    # 5d. Append to progress.log\n    progress_path = restored_workspace / "progress.log"\n    with progress_path.open("a") as f:\n        f.write(json.dumps({\n            "event": "completed_on_host_b",\n            "host": host_label,\n            "platform": platform.platform(),\n            "timestamp": time.time(),\n            "epoch": 2,\n            "pipeline_output_hash": task_result["output_hash"],\n        }, sort_keys=True) + "\\n")\n\n    # 6. Seal Epoch 2 (with Host B ephemeral signing)\n    print("\\n[6/6] Sealing Epoch 2 successor capsule...")\n    capsule_epoch_2 = out_dir / "capsule_epoch_2"\n    if capsule_epoch_2.exists():\n        shutil.rmtree(capsule_epoch_2)\n\n    # Generate ephemeral Host B signing key\n    host_b_priv, host_b_pub = generate_keypair()\n\n    e2_manifest = seal_epoch_2(\n        restored_workspace,\n        capsule_epoch_2,\n        epoch=2,\n        parent_manifest_hash=verification["manifest_hash"],\n        host_label=host_label,\n        host_b_priv=host_b_priv,\n        host_b_pub=host_b_pub,\n        challenge_nonce=args.challenge_nonce,\n    )\n    print(f"  E2 manifest hash: {e2_manifest[\'manifest_hash\']}")\n    print(f"  E2 parent hash: {e2_manifest[\'parent_manifest_hash\']}")\n    print(f"  E2 workspace root: {e2_manifest[\'workspace_manifest\'][\'root_hash\']}")\n\n    # Create E2 transport tar\n    e2_tar = out_dir / "transport_capsule_epoch_2.tar.gz"\n    with tarfile.open(e2_tar, "w:gz") as tf:\n        for root, dirs, files in os.walk(capsule_epoch_2):\n            dirs.sort()\n            files.sort()\n            for fname in files:\n                fpath = Path(root) / fname\n                arcname = fpath.relative_to(capsule_epoch_2.parent).as_posix()\n                info = tarfile.TarInfo(name=arcname)\n                info.size = fpath.stat().st_size\n                info.mtime = 0\n                info.mode = 0o644\n                info.uid = 0\n                info.gid = 0\n                info.uname = ""\n                info.gname = ""\n                with fpath.open("rb") as f:\n                    tf.addfile(info, f)\n    e2_tar_bytes = e2_tar.read_bytes()\n    e2_tar_sha256 = sha256_bytes(e2_tar_bytes)\n\n    runner_end = time.time()\n    runner_end_utc = utc_now_iso()\n\n    # Write host_b_report.json\n    report = {\n        "schema": "hdar.host-b-report/v1.0",\n        "protocol_version": PROTOCOL_VERSION,\n        "host_b_identity": {\n            "host_label": host_label,\n            "machine_hostname": machine_hostname,\n            "platform": platform.platform(),\n            "python_version": sys.version,\n            "machine_nonce": machine_nonce,\n            "operator": args.operator,\n            "runner_start_utc": runner_start_utc,\n            "runner_end_utc": runner_end_utc,\n            "runner_duration_seconds": round(runner_end - runner_start, 3),\n        },\n        "host_b_platform": platform.platform(),\n        "host_a_platform": EMBEDDED_HOST_A_PLATFORM,\n        "platforms_differ": platform.platform() != EMBEDDED_HOST_A_PLATFORM,\n        "capsule_e1_verification": verification,\n        "workspace_restoration": restore_report,\n        "pipeline_result": task_result,\n        "capsule_e2": {\n            "manifest_hash": e2_manifest["manifest_hash"],\n            "parent_manifest_hash": e2_manifest["parent_manifest_hash"],\n            "workspace_root_hash": e2_manifest["workspace_manifest"]["root_hash"],\n            "file_count": len(e2_manifest["workspace_manifest"]["files"]),\n            "total_size": e2_manifest["workspace_manifest"]["total_size"],\n            "epoch": 2,\n        },\n        "transport_capsule_e2": {\n            "bytes": len(e2_tar_bytes),\n            "sha256": e2_tar_sha256,\n        },\n        "owner_public_key": EMBEDDED_OWNER_PUB,\n        "host_b_public_key": host_b_pub.hex(),\n        "host_b_signature": e2_manifest.get("host_b_signature", ""),\n        "challenge_nonce": args.challenge_nonce if args.challenge_nonce else None,\n        "claim": f"Host B ({host_label}) independently restored E1, verified owner signature, executed pipeline, updated agent state, sealed E2.",\n        "claim_boundary": "This report is generated by Host B. It proves the capsule was restorable and the pipeline is deterministic on this platform. Cross-platform proof requires the verifier to confirm platforms_differ=true.",\n    }\n    report_path = out_dir / "host_b_report.json"\n    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\\n")\n\n    print("\\n" + "=" * 60)\n    print(f"Host B complete. Report: {report_path}")\n    print("=" * 60)\n    print(f"  Platform: {platform.platform()}")\n    print(f"  Platform separation: {report[\'platforms_differ\']}")\n    print(f"  Pipeline output hash: {task_result[\'output_hash\']}")\n    print(f"  E2 manifest hash: {e2_manifest[\'manifest_hash\']}")\n    print(f"  Duration: {round(runner_end - runner_start, 3)}s")\n    print()\n    print("To verify: copy host_b_report.json + capsule_epoch_2/ back to Host A")\n    print("           then run: python3 verify_all.py --host-a-dir <host_a_dir> --host-b-report <report.json> --e2-capsule <capsule_epoch_2>")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
with open('run_host_b.py', 'w') as f:
    f.write(runner_code)
print(f'Runner written: {{len(runner_code)}} bytes')

## Run Host B

This restores the capsule, verifies the signature, executes the pipeline, and seals Epoch 2.

In [ ]:
!python3 run_host_b.py --out ./host_b_output --host-label colab-$(hostname) --operator colab-user

## Show the Host B report

In [ ]:
import json
with open('./host_b_output/host_b_report.json') as f:
    report = json.load(f)
print(json.dumps(report, indent=2))

## Serve evidence via ngrok tunnel

Starts a local HTTP server on the Colab VM and exposes it via ngrok.
The tunnel URL is printed below — CI uses it to download evidence directly.
This replaces files.download() which doesn't work in automated browser sessions.

In [ ]:
!pip install pyngrok -q
import shutil, os, http.server, threading, json
from pyngrok import ngrok

# Pack E2 capsule as tarball
shutil.make_archive('/tmp/capsule_epoch_2', 'gztar', './host_b_output/capsule_epoch_2')

# Copy evidence files to a serve directory
serve_dir = '/tmp/hdar_evidence'
os.makedirs(serve_dir, exist_ok=True)
shutil.copy('./host_b_output/host_b_report.json', f'{serve_dir}/host_b_report.json')
shutil.copy('/tmp/capsule_epoch_2.tar.gz', f'{serve_dir}/capsule_epoch_2.tar.gz')

# Write a manifest so CI knows what's available
manifest = {
    'files': ['host_b_report.json', 'capsule_epoch_2.tar.gz'],
    'platform': __import__('platform').platform(),
    'hostname': __import__('socket').gethostname(),
}
with open(f'{serve_dir}/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# Start a simple HTTP server in a background thread
os.chdir(serve_dir)
server = http.server.HTTPServer(('0.0.0.0', 8080), http.server.SimpleHTTPRequestHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print('HTTP server started on port 8080')

# Open ngrok tunnel
tunnel = ngrok.connect(8080)
tunnel_url = tunnel.public_url
print(f'HDAR_TUNNEL_URL={tunnel_url}')
print(f'Evidence available at: {tunnel_url}/manifest.json')
print(f'Report: {tunnel_url}/host_b_report.json')
print(f'Capsule: {tunnel_url}/capsule_epoch_2.tar.gz')